# GitHub-hosted Model Registration Demo

This notebook shows how to package a scikit-learn classifier from a GitHub repository,
build a wheel, publish it (simulated via local artifact), register the configuration with
Model Zoo, and execute a run end to end.


## 1. Clone the GitHub repository

Replace `https://github.com/your-org/marketing-model.git` with your actual repository.
The repo is expected to contain the model code under `src/` and a minimal `pyproject.toml`.


In [ ]:
%%bash
set -euxo pipefail
REPO_URL="https://github.com/your-org/marketing-model.git"
if [ ! -d github_model ]; then
  git clone "$REPO_URL" github_model
else
  cd github_model
  git fetch --all
  git checkout main
  git pull
fi
ls github_model


## 2. Ensure a callable entry point exists

For demonstration, we create `src/github_marketing/predict.py` exposing `run_model`.
In your real project, ensure the repository already contains this function.


In [ ]:
%%bash
set -euxo pipefail
mkdir -p github_model/src/github_marketing
cat <<'PY' > github_model/src/github_marketing/predict.py
from __future__ import annotations
from typing import Any, Dict
import numpy as np
from sklearn.linear_model import LogisticRegression

def run_model(inputs: Dict[str, Any], params: Dict[str, Any], context: Any) -> Dict[str, Any]:
    X = np.asarray(inputs['X'], dtype=float)
    y = np.asarray(inputs['y'], dtype=int)
    solver = params.get('solver', 'lbfgs')
    max_iter = int(params.get('max_iter', 200))
    clf = LogisticRegression(solver=solver, max_iter=max_iter)
    clf.fit(X, y)
    predictions = clf.predict(X).tolist()
    probs = clf.predict_proba(X).tolist()
    diagnostics = {
        'coef': clf.coef_.tolist(),
        'intercept': clf.intercept_.tolist(),
        'classes': clf.classes_.tolist(),
    }
    return {
        'predictions': predictions,
        'artifacts': {'probabilities': probs},
        'diagnostics': diagnostics,
    }
PY


## 3. Build the wheel

Install the build tooling if necessary and produce a `.whl` artifact.
In production you would publish this wheel to your package registry (PyPI, Artifactory, etc.).


In [ ]:
%%bash
set -euxo pipefail
python -m pip install --upgrade pip build
cd github_model
python -m build --wheel --outdir ../dist
ls -1 ../dist


## 4. Configure Model Zoo

Create the configuration document referencing the GitHub package and the module/callable we just defined.


In [ ]:
import json

config_payload = {
    'model': {
        'id': 'github_logistic_v1',
        'name': 'GitHub Logistic Classifier',
        'version': '1.0.0',
        'kind': 'library',
        'binding': {
            'module': 'github_marketing.predict',
            'callable': 'run_model'
        },
        'inputs': ['X', 'y']
    },
    'dependencies': [
        'github-marketing==1.0.0',
        'scikit-learn>=1.4',
        'numpy>=1.24'
    ],
    'hyperparameters': {
        'schema': [
            {'name': 'solver', 'dtype': 'string', 'default': 'lbfgs'},
            {'name': 'max_iter', 'dtype': 'int', 'default': 200}
        ],
        'values': [
            {'model_id': 'github_logistic_v1', 'param': 'solver', 'value': 'lbfgs'},
            {'model_id': 'github_logistic_v1', 'param': 'max_iter', 'value': '250'}
        ]
    },
    'variables': {
        'definitions': [
            {'name': 'X', 'source': 'feature_matrix'},
            {'name': 'y', 'source': 'labels'}
        ]
    },
    'context': {'timeout_s': 180}
}

print(json.dumps(config_payload, indent=2))


## 5. Register the configuration

POST the configuration to the registry service.
Before running this cell, ensure `docker compose up` has started the API and Postgres services.


In [ ]:
import requests

registry_url = 'http://localhost:8000/registry/configs'
response = requests.post(registry_url, json={'env': 'prod', 'config': config_payload}, timeout=30)
response.raise_for_status()
response.json()


## 6. Generate synthetic data

Create a toy dataset for the logistic regression model.


In [ ]:
import numpy as np

rng = np.random.default_rng(123)
X = rng.normal(size=(50, 2))
beta = np.array([1.2, -0.7])
logits = X @ beta + 0.5
probabilities = 1.0 / (1.0 + np.exp(-logits))
y = (probabilities > 0.5).astype(int)
feature_matrix = X.round(4).tolist()
labels = y.tolist()


## 7. Submit a run

Call the `/runs` endpoint. The registry installs dependencies, imports the GitHub package,
and executes the callable.


In [ ]:
run_payload = {
    'model_id': config_payload['model']['id'],
    'env': 'prod',
    'data_sources': {
        'feature_matrix': feature_matrix,
        'labels': labels
    },
    'params': {'solver': 'lbfgs', 'max_iter': 250},
    'context': {'timeout_s': 180}
}

run_response = requests.post('http://localhost:8000/runs', json=run_payload, timeout=30)
run_response.raise_for_status()
ticket = run_response.json()
ticket


## 8. Retrieve the results

Fetch predictions, probabilities, and diagnostics from the completed run.


In [ ]:
status_resp = requests.get(f"http://localhost:8000/runs/{ticket['run_id']}", timeout=30)
status_resp.raise_for_status()
result_payload = status_resp.json()
result_payload['result']
